## PREPARING DATA

In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from matplotlib.ticker import MultipleLocator


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent
    print(script_dir)
    data_folder = script_dir /'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG_Original = timeSeriesData_BIGraw.copy()
userInputData_Original = userInputDataRaw.copy()

In [ ]:
userInputData_Original

In [ ]:
userInputData_Original["room"].unique()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space d
room_mask = userInputData_Original["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'])
mask = room_mask  
userInputData_Original = userInputData_Original.loc[mask]
timeSeriesData_BIG_Original = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.index)]

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
column_to_transform = "x-y axis"
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                    for id in range(3)]

userInputData_Original.loc[:,sensors_position_columns] = userInputData_Original.loc[:,sensors_position_columns].applymap(lambda lst: [round(x,2) for x in lst])
userInputData_Original.loc[:,sensors_euclidian_distance_columns] = userInputData_Original.loc[:,sensors_euclidian_distance_columns].round(2)


In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
column_to_transform = sensors_position_columns
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [ ]:
userInputData_Original

In [ ]:
userInputData_Original.columns

In [ ]:

euclidian_distance_columns = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                               for id in range(3) ]

mask = userInputData_Original["are-doors-opened"] == "on"
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] == on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")
mask = userInputData_Original["are-doors-opened"] != "on"    
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] != on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]=="on"].shape

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]!="on"].shape

In [ ]:
max_before= -30
max_after = 299

column_to_check = "time taken before insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] >max_before
print(f"{column_to_check} < {max_before}:\n {userInputData_Original.loc[mask,column_to_check]}")

column_to_check = "time taken after insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] < max_after
print(f"{column_to_check} < {max_after}:\n {userInputData_Original.loc[mask,column_to_check]}")
print(f"userInputData_Original rows {userInputData_Original.shape[0]}")

In [ ]:
userInputData_Original.columns

# LIBRARIES AND FUNCTIONS DECLARACTIONS

## LIBRARIES

In [ ]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClcassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score

import math
from itertools import combinations
from sklearn.ensemble import StackingClassifier
from sklearn.preprocessing import LabelEncoder


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report



## MODELS

In [ ]:

# 1. Logistic Regression
logreg = LogisticRegression(max_iter=5000)
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],          # 'l1' requires solver='liblinear' or 'saga'
    "solver": ["lbfgs", "saga"]
}

# 2. Support Vector Classifier
svc = SVC()
svc_params = {
    "kernel": ["rbf", "poly", "sigmoid"],
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto"]
}

# 3. K-Nearest Neighbors
knn = KNeighborsClassifier()
knn_params = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

# 4. Decision Tree Classifier
dt = DecisionTreeClassifier()
dt_params = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 5. Random Forest Classifier
rf = RandomForestClassifier(random_state=42)
rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 6. Gradient Boosting Classifier
gbc = GradientBoostingClassifier()
gbc_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 7. Neural Network (MLPClassifier)
mlp = MLPClassifier(max_iter=3000)
mlp_params = {
    "hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01]
}


# Combine them into your expected dictionary:
models = {
    "LogisticRegression": (logreg, logreg_params),
    "SVC": (svc, svc_params),
    "KNN": (knn, knn_params),
    "DecisionTree": (dt, dt_params),
    "RandomForest": (rf, rf_params),
 #   "GradientBoosting": (gbc, gbc_params),
    "MLPClassifier": (mlp, mlp_params),
}



## FUNCTIONS

#### get_16_train_positions

In [ ]:
def get_16_train_positions(userInputData_Original:pd.DataFrame):
    column_to_check = "x-y axis"
    mask = (userInputData_Original["are-doors-opened"] != "on") & (userInputData_Original["room"] =='Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' ) & (userInputData_Original[column_to_check] != (None,None))
    training_positions = userInputData_Original.loc[mask,column_to_check].unique()
    return training_positions


#### get_Data_To_Be_Used

In [ ]:
def get_Data_To_Be_Used(userInputData_Original:pd.DataFrame,timeSeriesData_BIG_Original:pd.DataFrame,
                        keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)->(pd.DataFrame,pd.DataFrame):

    userInputData = userInputData_Original.copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.copy()

    # clear the data that doesn't fit to the 330 columns we want
    max_before= -30
    max_after = 299
    
    mask_before = (userInputData["time taken before insertion (seconds)-capped"] == max_before)
    max_after = (userInputData["time taken after insertion (seconds)-capped"] == max_after)
    mask = mask_before & max_after
    print(f"data which doesn't fit our criteria of size:{userInputData.loc[~mask,:].index}")
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    if keepOpenDoorData:
        mask = (userInputData["are-doors-opened"] == "on") | (userInputData["are-doors-opened"] != "on")

    else:
        mask =  userInputData["are-doors-opened"] != "on"

    if anyOtherMaskToBeUsed is not None:
        mask = mask & anyOtherMaskToBeUsed
    userInputData = userInputData.loc[mask].copy()
    #grab all the data that are contained in those experiments
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
        
    
    if drop_other_positions & (not closeDoorTraining_openDoorTest):
        #from the experiments with door open, drop the positions which doesn't fit to the 16 source position we are going to check the model
        #also drop the experiments with the None values
        axis_list = get_16_train_positions(userInputData_Original)
        axis_mask = userInputData["x-y axis"].isin(axis_list)
        
        mask = axis_mask 
        userInputData = userInputData.loc[mask]
        #grab all the data that are contained in those experiments
        timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

    return userInputData,timeSeriesData_BIG
        

#### build_X_keys_And_Y_target_Arrays

In [ ]:
def build_X_keys_And_Y_target_Arrays(userInputData,closeDoorTraining_openDoorTest = False)->(np.array,np.array):
  
    if closeDoorTraining_openDoorTest :
         pass

    else:
        keys = np.array(userInputData.index)
        keys_numpy = keys.reshape(-1,1)
   

        # columns_to_pass = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
        #                            for id in range(3) ]
        # column_size = len(columns_to_pass)
        # target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        # target_variables_euclidian_distance_numpy = target_variables.reshape(-1,column_size)
     
    
    
        columns_to_pass = ["x-y axis"]
        flattened = userInputData.loc[keys,columns_to_pass].to_numpy().ravel()
        Label_Encoder = LabelEncoder()
                # Step 1: flatten the array from shape (98, 1) of tuples to (98,) of tuples
        
        # Step 2: convert each tuple to a string "x_y"
        string_labels = np.array([f"{x}_{y}" for (x, y) in flattened])
        
        # Step 3: reshape to (98, 1)
        string_labels = string_labels.reshape(-1, 1)
        target_variables_positions_numpy =  Label_Encoder.fit_transform(string_labels) 
        
        
        X_train_keys,X_test_keys,Y_train,Y_test = train_test_split(
            keys_numpy,target_variables_positions_numpy,
            test_size=0.2,
            stratify=target_variables_positions_numpy,
            random_state=42
        )
    
    return X_train_keys,X_test_keys,Y_train,Y_test,Label_Encoder

In [ ]:
def build_X_from_dataframe(keys_array, df, column_to_take):
    # flatten [[159],[196]...] → [159,196,...]
    keys = keys_array.ravel()

    # number of keys
    N = len(keys)

    # number of time steps per experiment (should be constant)
    example_key = keys[0]
    time_values = df[df["keys"] == example_key]
    T = len(time_values)

    # allocate X
    X = np.zeros((N, T))

    # fill X
    for i, key in enumerate(keys):
        rows = df[df["keys"] == key]
        X[i, :] = rows[column_to_take].values

    return X

In [ ]:
def get_Data_Per_Sensor(userInputData,timeSeriesData_BIG) ->pd.DataFrame:
    df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    sensors_names = np.sort(timeSeriesData_BIG["sensors"].unique())
    dfs_by_sensor = {
        sensor: df_filtered.loc[df_filtered["sensors"]==sensor]
        for sensor in sensors_names
    }
    return dfs_by_sensor


In [ ]:
def build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take ="VOC rolling average")->(np.array,np.array):
    dfs_by_sensor = get_Data_Per_Sensor(userInputData,timeSeriesData_BIG)
    X_train = np.empty((len(dfs_by_sensor),X_train_keys.shape[0],330))
    
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_train[i,:,:]= build_X_from_dataframe(X_train_keys,data_per_sensor,column_to_take)
    
    X_test = np.empty((len(dfs_by_sensor),X_test_keys.shape[0],330))
        
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_test[i,:,:]= build_X_from_dataframe(X_test_keys,data_per_sensor,column_to_take)

    return X_train,X_test

#### create_The_Arrays_For_The_Models

In [ ]:

def create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take ="VOC rolling average",keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None):
    userInputData,timeSeriesData_BIG = get_Data_To_Be_Used(userInputData_Original,timeSeriesData_BIG_Original,
                                                           keepOpenDoorData,drop_other_positions,closeDoorTraining_openDoorTest,anyOtherMaskToBeUsed)
    
    X_train_keys,X_test_keys,Y_train,Y_test,LabelEncoder = build_X_keys_And_Y_target_Arrays(userInputData)    
    
    X_train,X_test = build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take)
    if apply_functions_to_X_data is not None and apply_functions_to_X_data is not np.nan:
       for modify_function in apply_functions_to_X_data:
           if modify_function is not None and modify_function is not np.nan:
               X_train,X_test =  modify_function(X_train,X_test)
    return X_train,X_test,Y_train,Y_test,LabelEncoder ,userInputData,timeSeriesData_BIG

In [ ]:

    
def findPCAcomponentsCovered(X_train):     
    
    for X_train_per_sensor in X_train:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train_per_sensor)
        # Step 2: Apply PCA
        pca = PCA()
        X_pca = pca.fit_transform(X_scaled)
        
        # Step 3: Explained variance
        explained_variance = pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)
        
        # Only display first 10 components max
        max_components = min(10, len(explained_variance))
        ev_to_plot = explained_variance[:max_components]
        cum_to_plot = cumulative_variance[:max_components]
        
        # Step 4: Bar plot of cumulative explained variance
        plt.figure(figsize=(8,5))
        plt.bar(range(1, max_components + 1), cum_to_plot)
        plt.xlabel('Number of PCA components')
        plt.ylabel('Cumulative explained variance')
        plt.title('Cumulative explained variance (first 10 components)')
        plt.grid(True, axis='y')
        plt.show()
        
        # Step 5: Optimal number of components for ~90% variance
        optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
        print("Optimal number of components to explain ~90% variance:", optimal_components)


In [ ]:
def printPCA2components(X_train,Y_train):
    for i in range(X_train.shape[0]):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train[i])
        
        # --- Step 2: Reduce to 2 components ---
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        
        # --- Step 3: Scatter plot with hue based on Y_train ---
        plt.figure(figsize=(8,6))
        
        # Using seaborn for easy color scaling
        sns.scatterplot(
            x=X_pca[:,0], 
            y=X_pca[:,1], 
            hue=Y_train_Euclidians[:,i],              # color represents Y_train
             palette="coolwarm",         # continuous color palette
            s=80,                      # marker size
            alpha=0.8
        )
        
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.title("2D PCA of X_train colored by Y_train")
        plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.show()

#### run_grid_search_per_model

In [ ]:

def run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params):
        PCA__n_components = [2,3,4,5,6,8,10]

        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']

        results = {}
        if name in estimators_with_no_scaling_need:
             pipe = Pipeline([
                 ('model', model)
             ]) 
        else:
             pipe = Pipeline([
                 ('scaler', StandardScaler()),
                 ('PCA', PCA()),
                 ('model', model)
             ])
        # Build a correct param grid:
        param_grid = {
            **{'model__' + k: v for k, v in params.items()}
        }
        if name not in estimators_with_no_scaling_need:
            param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='accuracy',
            verbose=2
        )
     
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor(X_train, y_train,cv_number,verbose,models):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
     #       print(best_result["name"])
            best_result["score"] = results["score"]
     #       print(best_result["score"])
            best_result["model"] = results["model"]
       #     print(best_result["model"])
            best_result["parameters"] = results["parameters"]
       #     print(best_result["parameters"])
    return best_result


def run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, y_train,cv_number,verbose,models):

    best_results_for_all_sensors = {}

    for i in range(X_train.shape[0]):
        best_results_for_all_sensors[i]  =run_grid_search_find_optimal_model_per_sensor(X_train[i,:,:], y_train[:,AXIS],cv_number,verbose,models)

    return best_results_for_all_sensors

#### find_best_stacked_regression_Pre_Given

In [ ]:
def find_best_stacked_Estimator_Pre_Given(X_train, y_train,cv_number,verbose,models,best_result_per_sensor,AXIS):
    
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {} 
    }
    
    for final_estimator_name, (final_estimator_model, final_estimator_params) in models.items():
        for base_best_result in best_result_per_sensor.values():
            
            base_estimators = [(str(sensor_names),base_best_result['model']) 
                               for sensor_names,base_best_result in best_result_per_sensor.items()]
    
       # print(f"base_estimators{base_estimators}")
        stacked_Estimator = StackingRegressor(
            estimators=base_estimators,
            final_estimator=final_estimator_model,  # we will tune this
            passthrough=True           # include original features if you want
        )
        param_grid = {
            **{'final_estimator' +'__' + k: v for k, v in final_estimator_params.items()}
        }
        print(X_train[0,:,:].shape)
        print(y_train[:,AXIS].shape)
        stacked_grid = GridSearchCV(stacked_Estimator,param_grid,cv=cv_number,verbose=verbose,n_jobs=-1,scoring='accuracy')
        stacked_grid.fit(X_train[0,:,:], y_train[:,AXIS])
    
        
        if stacked_grid.best_score_ > best_result["score"]:
                best_result["name"]  = final_estimator_name
                best_result["model"] = stacked_grid.best_estimator_
                best_result["parameters"] = stacked_grid.best_params_
                best_result["score"] = stacked_grid.best_score_ 
    return  best_result           

#### getPredictedValues

In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
def getPredictedValues(X_test,Y_test,best_results_for_all_sensors,is_Stacked = False):


   if (is_Stacked is False): 
       predict_Y = np.empty((X_test.shape[0],Y_test.shape[0],Y_test.shape[1]))
        
       for i in range(X_test.shape[0]):
            print(f"For sensor with id={i}")
            predict_Y[i,:,:] = best_results_for_all_sensors[i]["model"].predict(X_test[i,:,:])
   else:
        predict_Y = np.empty((Y_test.shape[0],Y_test.shape[1]))
        predict_Y = best_results_for_all_sensors["model"].predict(X_test[0,:,:])

   return predict_Y
       

####

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train,Y_test,LabelEncoder,models):
    cv_number = 5
    verbose = 0
    best_results_for_all_sensors = {}
    best_results_for_all_sensors= run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, Y_train,cv_number,verbose,models)
    
    Y_predict_Base = getPredictedValues(X_test,Y_test,best_results_for_all_sensors,is_Stacked = False)
    
    for Y_predict in Y_predict_Base:
         plotConfusionMatrix(Y_test, Y_predict,LabelEncoder)
         accuracy,precision,recall   = getScores(Y_test, Y_predict)  
         print(f"accuracy:{accuracy},precision:{precision},recall:{recall}")
    return Y_predict_Base,best_results_for_all_sensors

#### trainModels_PrintResults_AndGetPredictedValues_Stacked

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues_Stacked(userInputData,best_results_for_all_sensors, X_train,X_test,Y_train,Y_test,Label_Encoder,models):
    cv_number = 5
    verbose = 0
    best_results_stacked = {}
    best_results_stacked = find_best_stacked_Estimator_Pre_Given(X_train, Y_train,cv_number,verbose,models,best_results_for_all_sensors)
    
    
    Y_predict_Stacked = getPredictedValues(X_test,Y_test,best_results_stacked,is_Stacked = True)
    
    plotConfusionMatrix(Y_test, Y_predict_Stacked,Label_Encoder)
    accuracy,precision,recall   = getScores(Y_test, Y_predict_Stacked) 
    
    return Y_predict_Stacked,best_results_stacked, accuracy,precision,recall 

#### grabPositionOfSensors

In [ ]:
def grabPositionOfSensors(userInputData):
 # --- SENSOR POINTS (red true, blue/orange predicted) ---
    sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
    point_of_sensors = []
    mask = userInputData["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    for sensors_position_column in sensors_position_columns:
        
        point_of_sensors.append(userInputData.loc[mask,sensors_position_column].iloc[0]) 

    return   point_of_sensors 

#### PLOT CONFUSION MATRIX  

In [ ]:
def plotConfusionMatrix(y_test, y_pred,Label_Encoder):
    # encoder used earlier
    labels = le.classes_                # encoded labels in correct order
    decoded_labels = [str(x) for x in labels]  # tuples → strings for display
    
    cm = confusion_matrix(y_test, y_pred, labels=range(len(labels)))
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=decoded_labels
    )
    
    disp.plot(cmap='Blues', values_format='d')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

#### GET SCORES

In [ ]:
def getScores(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    
    precision = precision_score(
        y_test, y_pred,
        average='weighted'   # or 'macro', 'micro'
    )
    
    recall = recall_score(
        y_test, y_pred,
        average='weighted'
    )
    
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    return accuracy,precision,recall

#### PIPELINE

In [ ]:

def runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params ={},apply_functions_to_X_data = []):

    (X_train,X_test,Y_train,Y_test,LabelEncoder,userInputData,
        timeSeriesData_BIG) = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data,column_to_take,**extra_params)
    
    #printPCA2components(X_train,Y_train)
    
    Y_predict_Base,best_model_per_second = trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train,Y_test,LabelEncoder,models)
    Y_predict_Stacked,best_model, accuracy,precision,recall = trainModels_PrintResults_AndGetPredictedValues_Stacked(userInputData,best_model_per_second, X_train,X_test,Y_train,Y_test,LabelEncoder,models)



    return  (X_train,X_test,Y_train,Y_test, 
            userInputData,timeSeriesData_BIG,  
            Y_predict_Base,best_model_per_second , Y_predict_Stacked,best_model,
            accuracy,precision,recall)

In [ ]:
def circle_intersections(c1, r1, c2, r2):
    x0, y0 = c1
    x1, y1 = c2

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    # No intersection
    if d > r1 + r2 or d < abs(r1 - r2) or d == 0:
        return []

    # Compute a, h
    a = (r1**2 - r2**2 + d**2) / (2*d)
    h = math.sqrt(max(0, r1**2 - a**2))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    xs1 = xm + h * dy / d
    ys1 = ym - h * dx / d
    xs2 = xm - h * dy / d
    ys2 = ym + h * dx / d

    return [(xs1, ys1), (xs2, ys2)]

# ------------------------------------------------------------
# Collect intersection points that lie inside all circles
# ------------------------------------------------------------
def intersection_points_of_all_circles(points, radii, tol=1e-6):
    candidates = []

    # pairwise intersections
    for (p1, r1), (p2, r2) in combinations(zip(points, radii), 2):
        pts = circle_intersections(p1, r1, p2, r2)
        candidates.extend(pts)

    # keep only points that lie inside ALL circles
    inside = []
    for x, y in candidates:
        if all(math.hypot(x - px, y - py) <= r + tol
               for (px, py), r in zip(points, radii)):
            inside.append((x, y))

    return inside

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------
def plot_circles_and_intersections(points, radii,i,at):
    fig, ax = plt.subplots(figsize=(6, 6))

    # Draw all circles
    for (px, py), r in zip(points, radii):
        circle = plt.Circle((px, py), r, fill=False, lw=2)
        ax.add_patch(circle)
        ax.plot(px, py, 'ko')  # centers

    # Compute intersections
    ints = intersection_points_of_all_circles(points, radii)

    # Plot intersection points
    for x, y in ints:
        ax.plot(x, y, 'ro', markersize=8)

    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_title("Circles and Intersection Points for row " + str(i) + " at "+at)
    plt.show()

## FILTER FUNCTIONS

In [ ]:
def cutOutliers(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], q25, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], q25, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], q25, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], q25, q75)        

    return X_train,X_test       

In [ ]:
def cutOutliersUpper(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], None, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], None, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersUpperPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], None, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], None, q75)        

    return X_train,X_test       

In [ ]:
def createGradient(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,j,:] = np.gradient(X_train[i,j,:])
            
    for i in range(X_test.shape[0]):
        for j in range(X_test.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_test[i,j,:] = np.gradient(X_test[i,j,:])        

    return X_train,X_test       

# FIND THE VARIABLES

## first type


for i in range(X_train.shape[0]):
    scaler = StandardScaler()
    cutOutliers()
    X_scaled = scaler.fit_transform(X_train[i])
    
    # --- Step 2: Reduce to 2 components ---
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # --- Step 3: Scatter plot with hue based on Y_train ---
    plt.figure(figsize=(8,6))
    
    # Using seaborn for easy color scaling
    sns.scatterplot(
        x=X_pca[:,0], 
        y=X_pca[:,1], 
        hue=Y_train_Euclidians[:,i],              # color represents Y_train
        palette="viridis",         # continuous color palette
        s=80,                      # marker size
        alpha=0.8
    )
    
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title("2D PCA of X_train colored by Y_train")
    plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.show()

## CREATE DATAFRAMES AND FUNCTION TO PRINT

In [ ]:
functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
functions_to_apply_Gradient = [None,createGradient]
functions_to_apply_Outliers_names = [X.__name__ if X is not None else X for X in functions_to_apply_Outliers]
functions_to_apply_Gradient_names = [X.__name__ if X is not None else X for X in functions_to_apply_Gradient]

column_to_take =["VOC original","VOC","VOC rolling average"]
room_cases = ["Only closed","All data"]
multi_index = pd.MultiIndex.from_product([room_cases,column_to_take,functions_to_apply_Gradient_names,functions_to_apply_Outliers_names],names= ["room cases" ,"column to take","Gradient functions","Outlier functions"])
for idx in multi_index:
    print(idx)

In [ ]:
columns = ["accuracy","precision","recall"]
parametersResultsGathered = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGathered

In [ ]:
def loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered):
    print(column_to_take)
    print(extra_params)
    functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
    functions_to_apply_Gradient = [None,createGradient]

    
    for function_Gradient in functions_to_apply_Gradient:
        gradient_name = function_Gradient.__name__ if function_Gradient is not None else np.nan

        for function_Outlier in functions_to_apply_Outliers:
            
            outlier_name = function_Outlier.__name__ if function_Outlier is not None else np.nan
            print("outlier_name "+str(outlier_name))
            print("gradient_name "+str(gradient_name))


            apply_functions_to_X_data = [function_Outlier,function_Gradient]

            (X_train,X_test,Y_train,Y_test, 
                userInputData,timeSeriesData_BIG,  
                Y_predict_Base,best_model_per_second , Y_predict_Stacked,best_model,
            accuracy,precision,recall) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

            idx = (room_case, column_to_take, gradient_name, outlier_name)
            parametersResultsGathered.loc[idx, ["accuracy","precision","recall"]] = [accuracy,precision,recall]
            print(parametersResultsGathered.loc[idx, ["accuracy","precision","recall"]])
    return parametersResultsGathered      
            # MSE,EUCLIDIAN,R2 the values i want to insert 

## RUN THE FUNCTIONS

### ONLY CLOSED

#### VOC Original

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC

In [ ]:
column_to_take = "VOC"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC rolling average

In [ ]:
column_to_take = "VOC rolling average"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
    key=lambda col: col.abs()
)
df_sorted

### ALL DATA

#### VOC Original

In [ ]:
column_to_take = "VOC original"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.head(60)

#### VOC

In [ ]:
column_to_take = "VOC"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.head(60)

In [ ]:
parametersResultsGathered.idmax()

#### VOC rolling average

In [ ]:
column_to_take = "VOC rolling average"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.head(60)

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
    key=lambda col: col.abs()
)
df_sorted